In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

class NeuralExplorer(nn.Module): 
    def __init__(self, state_dim, action_dim, hidden_dim=64, learning_rate=0.001):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
    
    def forward(self, state):
        logits = self.model(state)
        probs = torch.softmax(logits, dim=-1) 
        return probs
    
    def get_action_frequencies(self, state):
        with torch.no_grad():
            probs = self.forward(state)
        return probs.cpu().numpy()
    
    def get_least_tried_action(self, state):
        freqs = self.get_action_frequencies(state)
        least_tried_action = freqs.argmin()
        return least_tried_action
    
    def update(self, state, action):
        probs = self.forward(state)
        loss = -torch.log(probs[action] + 1e-10)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item() 

In [3]:
import gym

env = gym.make('CartPole-v1')
explorer = NeuralExplorer(state_dim=4, action_dim=2)

for episode in range(100):
    state, _ = env.reset()
    state = torch.tensor(state, dtype=torch.float32)
    
    done = False
    while not done:
        # Select action
        action = explorer.get_least_tried_action(state)
        
        # Take action in environment
        next_state, reward, done, _, _ = env.step(action)
        
        # Update explorer
        loss = explorer.update(state, int(action))
        
        state = torch.tensor(next_state, dtype=torch.float32)

/opt/anaconda3/envs/rl_env/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):
